# Project: Building Your First Vector Search System

Build a functional vector search system that demonstrates the core concepts: collections, points, similarity search, and filtering. You’ll design simple 4-dimensional vectors that represent different concepts or items.

In [1]:
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
import os

load_dotenv()

True

## Step 1: Initialize Client

In [2]:
client = QdrantClient(url=os.getenv("QDRANT_URL"))

## Step 2: Create Collection

In [3]:
collection_name = "day0_first_system"

In [4]:
# Delete the collection before create it
try:
    client.delete_collection(collection_name) 
except:
    pass

In [5]:
# RE-create the collection
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=4, distance=models.Distance.DOT),
)

# Create payload index right after creating the collection and before uploading any data to enable filtering.
# If you add it later, HNSW won't rebuild automatically—bump ef_construct (e.g., 100→101) to trigger a safe rebuild.
client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

## Step 3: Insert Points

**Dimensions**:

1. Affordability
2. Quality
3. Popularity
4. Innovation

In [6]:
points=[
    models.PointStruct(
        id=1,
        vector=[0.9, 0.1, 0.1, 0.8], # High affordability, high innovation
        payload={"name": "Budget Smartphone", "category": "electronics", "price": 299},
    ),
    models.PointStruct(
        id=2,
        vector=[0.2, 0.9, 0.8, 0.5], # High quality, high popularity
        payload={"name": "Bestselling Novel", "category": "books", "price": 19},
    ),
    models.PointStruct(
        id=3,
        vector=[0.8, 0.3, 0.2, 0.9], # High affordability, high innovation (similar to ID 1)
        payload={"name": "Smart Home Hub", "category": "electronics", "price": 89},
    ),
    models.PointStruct(
        id=4,
        vector=[0.3, 0.7, 0.9, 0.2],
        payload={"name": "iPone 17 Pro", "category": "electronics", "price": 1800},
    ),
    models.PointStruct(
        id=5,
        vector=[0.9, 0.2, 0.7, 0.2],
        payload={"name": "Samsung Galaxy A16", "category": "electronics", "price": 120},
    ),
    models.PointStruct(
        id=6,
        vector=[0.8, 0.5, 0.5, 0.8],
        payload={"name": "Regire Fux Stroke", "category": "misc", "price": 18},
    ),
    models.PointStruct(
        id=7,
        vector=[0.1, 0.7, 0.4, 0.9],
        payload={"name": "Garmin Griger Premiim", "category": "misc", "price": 20000},
    ),
]

client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

## Step 4: Test Searches

In [7]:
# Define a query vector for "affordable and innovative"
query_vector = [0.85, 0.2, 0.1, 0.9]

# 1. Basic similarity search
basic_results = client.query_points(collection_name, query=query_vector)

# 2. Filtered search (only find electronics)
filtered_results = client.query_points(
    collection_name,
    query=query_vector,
    # query_filter=models.Filter(must=[models.FieldCondition(key="category", match=models.MatchValue(value="electronics"))]),
    limit=3,
)
print("Filtered search results:")
for result in filtered_results.points:
    print("\t- ", result)

Filtered search results:
	-  id=3 version=3 score=1.5699999 payload={'name': 'Smart Home Hub', 'category': 'electronics', 'price': 89} vector=None shard_key=None order_value=None
	-  id=6 version=3 score=1.55 payload={'name': 'Regire Fux Stroke', 'category': 'misc', 'price': 18} vector=None shard_key=None order_value=None
	-  id=1 version=3 score=1.5149999 payload={'name': 'Budget Smartphone', 'category': 'electronics', 'price': 299} vector=None shard_key=None order_value=None
